# Baseline RAG query pipeline

Embed the user query → **wide vector search** on the S3 Vector index (`rag-vector-2`) → keep only hits whose **`source_line` falls in the first 100 lines** of the source JSONL → take the best **top‑k** by distance → resolve chunk text from **vector metadata** via `get_vectors()` when local JSONL lookup fails.

| Step | What happens |
|---|---|
| 1 | Same encoder as ingest (`EMBED_MODEL`, default `intfloat/multilingual-e5-small`) |
| 2 | Query prefixed with `query: ` (E5; corpus used `passage: ` in `full_embeddings.py`) |
| 3 | `query_vectors` with **top 100** candidates → keys + distances + metadata |
| 4 | **Filter** to `1 ≤ source_line ≤ 100`, sort by distance, keep `TOP_K` chunks |
| 5 | `get_vectors()` on unresolved keys → chunk text from vector metadata |

**Corpus skew.** The index is dominated by English↔Japanese parallel chunks (`eng_jap_*`, tens of thousands of JSONL lines). A naive top‑5 query almost always surfaces that file. Restricting to the **first 100 source lines** biases retrieval toward chunks defined early in each JSONL (where smaller grammar/style KBs often live entirely), while still allowing early‑file Eng↔Jap rows when they rank highly.

**Source line issues.** Treat **`source_line` in metadata** as the canonical **1-based line index** into the named `source_file` JSONL at ingest time. Vector **keys** (e.g. `file:8431`) may use a different convention than the printed `L8432` label; mismatches plus filename variants (`*_embedded_full.jsonl` vs `eng_jap_chunks.jsonl`) explain “unresolved” local reads even when the row exists.

**No local `kb/` files required** for final text if S3 metadata holds `text` / `chunk_text`. Local `kb/` is still used when paths align.

**Credentials:** `VECTORS_AWS_ACCESS_KEY_ID`, `VECTORS_AWS_SECRET_ACCESS_KEY`, and region (see `src/utils/aws_profiles.py`).


## 1. Dependencies (run once per environment)

In [9]:
%pip install sentence-transformers boto3 pandas --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Project path and imports

In [23]:
import json
import os
import sys
from pathlib import Path

import pandas as pd


def project_root() -> Path:
    """Directory that contains `src/` (works if cwd is repo root or `rag/`)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "retrieval").is_dir():
            return p
    raise RuntimeError("Could not find project root (folder with src/retrieval). cd to the repo root or rag/.")


_ROOT = project_root()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from src.retrieval.s3_vectors_rag import S3VectorsRAGRetriever

print(f"Project root: {_ROOT}")

Project root: c:\Users\Jonathan\Desktop\Y4S2\GenAI\is469


## 3. Configuration

Override with environment variables: `RAG_VECTOR_BUCKET`, `RAG_VECTOR_INDEX`, `RAG_KB_DIR`, `VECTORS_AWS_DEFAULT_REGION`, `EMBED_MODEL`.

In [25]:
# Final number of chunks passed to the prompt after line-band filtering.
TOP_K = 5
# Ask the index for more neighbors, then keep only chunks in the first MAX_SOURCE_LINE JSONL lines.
VECTOR_QUERY_TOP_K = 100
MAX_SOURCE_LINE = 100

VECTOR_BUCKET = os.environ.get("RAG_VECTOR_BUCKET", "is469-genai-grp-project")
INDEX_NAME = os.environ.get("RAG_VECTOR_INDEX", "rag-vector-2")
REGION = os.environ.get("VECTORS_AWS_DEFAULT_REGION", "ap-southeast-1")

_kb_env = os.environ.get("RAG_KB_DIR", "").strip()
KB_DIR = Path(_kb_env) if _kb_env else (_ROOT / "kb")

print(f"Bucket: {VECTOR_BUCKET}")
print(f"Index:  {INDEX_NAME}")
print(f"Region: {REGION}")
print(f"kb_dir: {KB_DIR} (exists={KB_DIR.is_dir()})")
print(f"vector query top_k={VECTOR_QUERY_TOP_K}, keep source_line 1..{MAX_SOURCE_LINE}, final TOP_K={TOP_K}")

Bucket: is469-genai-grp-project
Index:  rag-vector-2
Region: ap-southeast-1
kb_dir: c:\Users\Jonathan\Desktop\Y4S2\GenAI\is469\kb (exists=True)
vector query top_k=100, keep source_line 1..100, final TOP_K=5


## 3b. Build S3 Vectors client for chunk text resolution

Uses `s3vectors_client` (same credentials as the retriever) to call `get_vectors()` on the keys
returned by the query. Chunk text is resolved directly from the vector metadata stored in `rag-vector-2` —
no local `kb/` file or separate S3 bucket required.


In [26]:
from src.utils.aws_profiles import s3vectors_client

# Same bucket + index as the retriever
vectors_client = s3vectors_client(region_name=REGION)

def resolve_chunks_from_s3(keys: list[str]) -> dict[str, str]:
    """
    Fetch full vector metadata for a list of keys from rag-vector-2.
    Returns a dict of {key: chunk_text}.
    """
    if not keys:
        return {}
    response = vectors_client.get_vectors(
        vectorBucketName=VECTOR_BUCKET,
        indexName=INDEX_NAME,
        keys=keys,
        returnData=False,       # We only need metadata, not the embedding floats
        returnMetadata=True,
    )
    result = {}
    for vec in response.get("vectors", []):
        key = vec.get("key")
        meta = vec.get("metadata") or {}
        # Try common field names the ingest script may have used
        text = (
            meta.get("text")
            or meta.get("chunk_text")
            or meta.get("content")
        )
        if key and text:
            result[key] = text
    return result

print("✓ s3vectors_client ready — chunk text will be resolved from rag-vector-2 metadata")


✓ s3vectors_client ready — chunk text will be resolved from rag-vector-2 metadata


## 4. Build retriever (loads embedder on first `retrieve`)

The next cell replaces `s3_vectors_rag._guess_kb_paths` so metadata names match **this repo’s** `kb/` files (e.g. `eng_jap_chunks_embedded_full.jsonl` in metadata → `eng_jap_chunks_embedded.jsonl` on disk, and no double `.jsonl.jsonl` when metadata already includes `.jsonl`).

In [27]:
import src.retrieval.s3_vectors_rag as _s3_rag


def _guess_kb_paths(kb_dir: Path, source_file: str) -> list[Path]:
    """Map vector metadata `source_file` to JSONL paths under kb/ (repo layout)."""
    raw = (source_file or "").strip()
    if not raw:
        return []
    name = Path(raw).name
    if name.endswith(".jsonl"):
        basename = name
        stem_no_ext = Path(name).stem
    else:
        stem_no_ext = name
        basename = f"{name}.jsonl"
    candidates: list[Path] = []
    seen: set[Path] = set()

    def add(p: Path) -> None:
        if p not in seen:
            seen.add(p)
            candidates.append(p)

    add(kb_dir / basename)
    add(kb_dir / f"{stem_no_ext}_vectors.jsonl")
    if stem_no_ext.endswith("_embedded_full"):
        add(kb_dir / f"{stem_no_ext[:-len('_full')]}.jsonl")
    if stem_no_ext.endswith("_embedded") and not stem_no_ext.endswith("_embedded_full"):
        add(kb_dir / f"{stem_no_ext}_full.jsonl")

    return [p for p in candidates if p.is_file()]


_s3_rag._guess_kb_paths = _guess_kb_paths

retriever = S3VectorsRAGRetriever(
    vector_bucket_name=VECTOR_BUCKET,
    index_name=INDEX_NAME,
    region_name=REGION,
    kb_dir=KB_DIR,
    top_k=VECTOR_QUERY_TOP_K,
)

print("Retriever ready (encoder loads lazily on first retrieve).")

Retriever ready (encoder loads lazily on first retrieve).


## 5. Run retrieval

Set `USER_QUERY` to the student’s text, then run the next cell.

In [28]:
USER_QUERY = "Example: How do I express polite requests in Japanese?"

_, all_chunks = retriever.retrieve(USER_QUERY)


def _distance_sort_key(c):
    d = c.distance
    if d is None:
        return float("inf")
    return float(d)


# Keep only chunks whose ingest line lies in the first MAX_SOURCE_LINE rows (see notebook intro).
_in_band = [c for c in all_chunks if 1 <= c.source_line <= MAX_SOURCE_LINE]
_in_band.sort(key=_distance_sort_key)
chunks = _in_band[:TOP_K]

print(
    f"Vector pool: {len(all_chunks)} hits (top {VECTOR_QUERY_TOP_K}); "
    f"in source lines 1–{MAX_SOURCE_LINE}: {len(_in_band)}; using {len(chunks)} for context."
)
if len(chunks) < TOP_K:
    print(
        f"Note: fewer than TOP_K={TOP_K} chunks in the line band — "
        "increase MAX_SOURCE_LINE or VECTOR_QUERY_TOP_K if you need more context."
    )

# ── Resolve unresolved chunks from rag-vector-2 metadata ────────────────────
unresolved_keys = [
    c.key for c in chunks
    if (c.text or "").startswith("(no local text resolved")
]

if unresolved_keys:
    print(f"Resolving {len(unresolved_keys)} chunk(s) from rag-vector-2...")
    s3_text_map = resolve_chunks_from_s3(unresolved_keys)
    for chunk in chunks:
        if (chunk.text or "").startswith("(no local text resolved"):
            chunk.text = s3_text_map.get(chunk.key, chunk.text)

# ── Format context string ────────────────────────────────────────────────────
context = "\n---\n".join(
    f"[{c.source_file} L{c.source_line}]\n{c.text or '(unresolved)'}"
    for c in chunks
)

print("=== Formatted context (for prompting) ===\n")
print(context)

# ── DataFrame summary ────────────────────────────────────────────────────────
_preview_chars = 320
_rows: list[dict] = []
for i, c in enumerate(chunks, start=1):
    unresolved = (c.text or "").startswith("(no local text resolved")
    t = c.text or ""
    preview = t if len(t) <= _preview_chars else f"{t[: _preview_chars - 1]}…"
    _rows.append(
        {
            "rank": i,
            "distance": c.distance,
            "key": c.key,
            "source_file": c.source_file,
            "source_line": c.source_line,
            "resolved": not unresolved,
            "text_preview": preview,
            "chunk_text": t,
        }
    )

hits_df = pd.DataFrame(_rows)
if hits_df["distance"].notna().any():
    hits_df["distance"] = pd.to_numeric(hits_df["distance"], errors="coerce").round(6)

hits_summary = hits_df[
    ["rank", "distance", "key", "source_file", "source_line", "resolved", "text_preview"]
]
print("\n=== Retrieval hits (summary DataFrame: `hits_df`, display below) ===\n")
hits_summary


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector pool: 95 hits (top 100); in source lines 1–100: 3; using 3 for context.
Note: fewer than TOP_K=5 chunks in the line band — increase MAX_SOURCE_LINE or VECTOR_QUERY_TOP_K if you need more context.
=== Formatted context (for prompting) ===

[eng_jap_chunks_embedded_full.jsonl L79]
[1] EN: I'm tired.
[1] JA: 私は、疲れています。

[2] EN: Who wants some hot chocolate?
[2] JA: ホットチョコレート欲しい人ー？

[3] EN: Speak more slowly, please!
[3] JA: もっとゆっくり話してください！

[4] EN: When do we arrive?
[4] JA: いつ着くの？

[5] EN: The check, please.
[5] JA: 勘定お願いします。

[6] EN: The check, please.
[6] JA: 勘定書をお願いします。

[7] EN: The check, please.
[7] JA: 勘定を頼むよ。

[8] EN: The check, please.
[8] JA: お勘定して下さい。
---
[eng_jap_chunks_embedded_full.jsonl L82]
[1] EN: I must admit that I snore.
[1] JA: いびきをかくことは認めるよ・・・。

[2] EN: Tonight we're going to church.
[2] JA: 今夜教会に行くよ。

[3] EN: How are you? Did you have a good trip?
[3] JA: 元気？旅行は良かった？

[4] EN: I don't feel well.
[4] JA: 気分がよくないんだ。

[5] EN: Call the police!
[5] JA: 警察を呼んで！

[6]

,rank,distance,key,source_file,source_line,resolved,text_preview
0,1,0.104292,eng_jap_chunks_embedded_full:78,eng_jap_chunks_embedded_full.jsonl,79,True,[1] EN: I'm tired.\n[1] JA: 私は、疲れています。\n\n[2] ...
1,2,0.107954,eng_jap_chunks_embedded_full:81,eng_jap_chunks_embedded_full.jsonl,82,True,[1] EN: I must admit that I snore.\n[1] JA: いび...
2,3,0.113325,eng_jap_chunks_embedded_full:79,eng_jap_chunks_embedded_full.jsonl,80,True,"[1] EN: The check, please.\n[1] JA: 勘定を頼むよ。\n\..."


In [29]:
# Source-line sanity check (Eng↔Jap skew + why local resolution disagrees with labels)
import json

chunks_file = KB_DIR / "eng_jap_chunks.jsonl"
print(f"{chunks_file.name} exists: {chunks_file.exists()}")

with open(chunks_file, encoding="utf-8") as f:
    lines = f.readlines()

print(f"Total JSONL lines: {len(lines):,} — raw top-k retrieval skews here; pipeline keeps only 1..{MAX_SOURCE_LINE}.\n")

print("First rows in the early band (typical context after filtering):")
for lineno in range(1, min(4, len(lines) + 1)):
    record = json.loads(lines[lineno - 1])
    snippet = json.dumps(record, ensure_ascii=False)[:180]
    print(f"  L{lineno}: {snippet}…")

# Deep rows exist on disk; "unresolved" in the retriever is usually path/name mismatch
# (metadata says eng_jap_chunks_embedded_full.jsonl but repo line lives in eng_jap_chunks.jsonl).
for lineno in [8432, 30252]:
    if lineno <= len(lines):
        record = json.loads(lines[lineno - 1])
        snippet = json.dumps(record, ensure_ascii=False)[:160]
        print(f"\nDeep row still valid at L{lineno} in eng_jap_chunks.jsonl: {snippet}…")

eng_jap_chunks.jsonl exists: True
Total JSONL lines: 46,616 — raw top-k retrieval skews here; pipeline keeps only 1..100.

First rows in the early band (typical context after filtering):
  L1: {"chunk_id": "engjap_chunk_000000_000007", "start_row": 0, "end_row": 7, "n_pairs": 8, "chunk_text": "[1] EN: Let's try something.\n[1] JA: 何かしてみましょう。\n\n[2] EN: I have to go to sl…
  L2: {"chunk_id": "engjap_chunk_000006_000013", "start_row": 6, "end_row": 13, "n_pairs": 8, "chunk_text": "[1] EN: The password is \"Muiriel\".\n[1] JA: パスワードは「Muiriel」です。\n\n[2] EN: I…
  L3: {"chunk_id": "engjap_chunk_000012_000019", "start_row": 12, "end_row": 19, "n_pairs": 8, "chunk_text": "[1] EN: I will be back soon.\n[1] JA: すぐ戻る。\n\n[2] EN: I'm at a loss for wor…

Deep row still valid at L8432 in eng_jap_chunks.jsonl: {"chunk_id": "engjap_chunk_050586_050593", "start_row": 50586, "end_row": 50593, "n_pairs": 8, "chunk_text": "[1] EN: I'm very pleased to meet you.\n[1] JA: お目に…

Deep row still valid at L30252